# TCG-pokebot — deck-inference experiments (single Save-&-Run-All)

Measures **how well `deck_inference.DeckGenerator` actually works** — the script that
guesses the opponent's hidden 60 during ISMCTS determinization. Three experiments, cheap→decisive:

* **C — structural stress checks** (no games): is the generator even functioning? Tag coverage,
  recipe/energy skew, world diversity vs matcher confidence.
* **A — offline reconstruction** (no games): hide a real decklist, reveal *k* cards, ask each method
  to reconstruct the rest, score *hidden-card recall* + *total-variation distance* vs baselines
  (`pool_sample`, `freq_fill`, `uniform_fill`, `matcher`). Tests whether a frequency/co-occurrence
  prior would beat the current uniform-within-role sampler — the crux of the "use a NN?" question.
* **B — in-search ablation** (real games, **decisive**): swap only the determinization source inside
  ISMCTS — `none` (floor) / `current` / `pool` / `oracle` (true deck, upper bound) — and measure the
  net's win-rate. **`oracle − current` is the headroom**: if perfect deck knowledge barely beats the
  current generator, no fancier model (NN included) can help and effort belongs elsewhere.

Needs the same inputs as the training notebook: the competition engine (`libcg.so`, `EN_Card_Data.csv`)
and — for **B only** — the weights dataset (`hakase0/tcg-pokebot-weights`) for a trained net.
Results are printed and written to `experiment_results.json`.

## 1. Config

In [ ]:
USERNAME        = 'hakase0'
REPO            = 'Hakase-0/TCG-pokebot'
CODE_DATASET    = ''                               # offline fallback: '<user>/tcg-pokebot-code'
WEIGHTS_DATASET = f'{USERNAME}/tcg-pokebot-weights'

# --- experiment knobs ---
SEED              = 0
# A (reconstruction)
RECON_REVEAL_KS   = [3, 6, 12, 20]   # cards "revealed" before reconstructing (early -> late game)
RECON_REPS        = 3                # random reveals per (deck,k); stochastic methods averaged over these
# B (in-search ablation) -- the decisive, costly one
ABLATION_KINDS    = ['none', 'current', 'pool', 'oracle']   # floor -> ... -> upper bound
ABLATION_PAIRS    = 40               # games PER kind (paired deck-pairs+seats shared across kinds)
ABLATION_WORLDS   = 3                # ISMCTS determinization worlds (match training: 3)
ABLATION_SIMS     = 24               # ISMCTS sims/world (lower than training's 48 to keep runtime sane)
ABLATION_LEAF     = 'value'          # leaf eval: 'value' (fast, needs trained head) | 'rollout' (slow)

import os, sys, glob, shutil, subprocess, time, json, random
WORKDIR = '/kaggle/working/TCG-pokebot'
def banner(t): print('\n' + '=' * 70 + f'\n  {t}\n' + '=' * 70, flush=True)
print('config OK | reveal-ks', RECON_REVEAL_KS, '| ablation', ABLATION_KINDS,
      f'{ABLATION_PAIRS}gx{ABLATION_WORLDS}w x{ABLATION_SIMS}s leaf={ABLATION_LEAF}')

## 2. Code (clone the repo)

In [ ]:
banner('SETUP: code')
def have_code(p): return os.path.exists(os.path.join(p,'ismcts.py')) and os.path.exists(os.path.join(p,'selfplay_rl.py'))
def _tok():
    try:
        from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception: return None
if not have_code(WORKDIR):
    done=False; tok=_tok()
    for url in [u for u in (f'https://{tok}@github.com/{REPO}' if tok else None, f'https://github.com/{REPO}') if u]:
        try:
            subprocess.run(['git','clone','--depth','1',url,WORKDIR],check=True); done=have_code(WORKDIR)
            if done: print('cloned from GitHub'); break
        except Exception as e: print('clone failed:', str(e)[:100])
    if not done and CODE_DATASET:
        src=f'/kaggle/input/{CODE_DATASET.split("/")[-1]}'
        cand=next((d for d,_,f in os.walk(src) if 'ismcts.py' in f),None)
        if cand:
            os.makedirs(WORKDIR,exist_ok=True)
            for f in glob.glob(os.path.join(cand,'*')):
                (shutil.copytree if os.path.isdir(f) else shutil.copy)(f,os.path.join(WORKDIR,os.path.basename(f)))
            done=have_code(WORKDIR); print('copied from code dataset:', done)
    assert done,'could not obtain code'
else: print('code already present')
os.chdir(WORKDIR); sys.path.insert(0,WORKDIR)
print('cwd', os.getcwd(), '|', len(glob.glob('*.py')),'py files')

## 3. Engine + tables + deck pool (offline)

In [ ]:
banner('SETUP: engine, tables, deck pool')
# engine from the competition input
if not os.path.exists(os.path.join(WORKDIR,'cg','libcg.so')):
    so=next((os.path.join(d,'libcg.so') for d,_,fs in os.walk('/kaggle/input') if 'libcg.so' in fs),None)
    assert so,'libcg.so not found under /kaggle/input — add the competition'
    shutil.copytree(os.path.dirname(so),os.path.join(WORKDIR,'cg'),dirs_exist_ok=True); print('engine copied')
if not os.path.exists('EN_Card_Data.csv'):
    src=next((os.path.join(d,'EN_Card_Data.csv') for d,_,fs in os.walk('/kaggle/input') if 'EN_Card_Data.csv' in fs),None)
    if src: shutil.copy(src,'EN_Card_Data.csv')
os.environ['CG_LIB_PATH']=os.path.join(WORKDIR,'cg')
from cg import api; print('engine OK — cards', len(api.all_card_data()))
# tables (capability_table.json + attack_table.json)
subprocess.run([sys.executable,'inspect_cards.py'],check=True,env={**os.environ})
print('tables:', os.path.exists('capability_table.json'))
# deck pool: txt -> id-csv (committed lists only; experiments stay deterministic/offline)
from import_deck import import_decklist, write_deck_csv
for txt in sorted(glob.glob('decks/*.txt'))+sorted(glob.glob('decks/adversary/*.txt')):
    csv=txt[:-4]+'.csv'
    if not os.path.exists(csv):
        try:
            ids,_=import_decklist(open(txt).read(),'EN_Card_Data.csv')
            if len(ids)==60: write_deck_csv(ids,csv)
        except Exception as e: print('skip',os.path.basename(txt),str(e)[:60])
print('deck pool:', len(glob.glob('decks/*.csv')),'meta +', len(glob.glob('decks/adversary/*.csv')),'adversary')

## 4. Load card context + net

Card DB, attack table, a `CardIndex` (what the generator derives its structure from), the real
deck pool, and — for **B only** — the newest trained net from the weights dataset. If no net is
attached, B is skipped automatically; A and C need no net.

In [ ]:
banner('LOAD: context + net')
import numpy as np, torch
from collections import Counter
import features as fx, arena, deck_inference as DI, policy_heuristic as H, ismcts
from evaluate import CardDB
from train_bc import device

DEV = device()
try:                                   # accelerator smoke test (P100/broken-wheel safety) -> CPU fallback
    torch.zeros(1, device=DEV).add_(1.0)
    if str(DEV) != 'cpu': torch.cuda.synchronize()
except Exception as e:
    print('accelerator smoke failed -> CPU:', str(e)[:80]); DEV = 'cpu'

db  = CardDB.load('capability_table.json')
atk = {int(k): v for k, v in json.load(open('attack_table.json')).items()}
index = DI.CardIndex(db)

def load_pool(pat):
    out=[]
    for f in sorted(glob.glob(pat)):
        d=[int(x) for x in open(f).read().split()][:60]
        if len(d)==60: out.append((os.path.basename(f)[:-4], d))
    return out
META = load_pool('decks/*.csv'); ADV = load_pool('decks/adversary/*.csv'); POOL = META + ADV
assert POOL, 'no decks imported — cannot run experiments'
print(f'device={DEV} | cards in db={len(index.entry)} | pool: {len(META)} meta + {len(ADV)} adv = {len(POOL)}')

def find_ckpt():
    roots=[f'/kaggle/input/{WEIGHTS_DATASET.split("/")[-1]}','/kaggle/working',WORKDIR]
    for pat in ('*.latest.pt','rl_model.pt','model.pt'):
        for r in roots:
            h=sorted(glob.glob(os.path.join(r,'**',pat),recursive=True),key=os.path.getmtime,reverse=True)
            if h: return h[0]
    return None
WARM = find_ckpt()
if WARM:
    md_src=os.path.join(os.path.dirname(WARM),'model_meta.json')
    if os.path.exists(md_src) and not os.path.exists('model_meta.json'): shutil.copy(md_src,'model_meta.json')
NET=None
if WARM:
    try: NET=arena.load_net(WARM,DEV); print('net loaded:', WARM)
    except Exception as e: print('net load FAILED -> B will skip:', str(e)[:120])
else:
    print('no checkpoint found -> B will skip (attach the weights dataset to run it)')
RESULTS={}

## 5. Experiment C — structural stress checks (cheapest)

Catches silent failures before we trust any number:
* **Tag coverage** — `_pool_for_role` falls back to "all trainers" (≈random) for any role whose
  cards aren't tagged. Low coverage ⇒ the recipe isn't really functioning.
* **Recipe / energy skew** — `_RECIPE` targets sum to <60, so padding fills the rest energy-first.
  Compare the empty-knowledge guess's energy count to real decks'.
* **Diversity vs confidence** — ISMCTS needs *diverse* worlds; check they don't collapse as the
  matcher gets confident.

In [ ]:
banner('EXPERIMENT C: structural stress checks')
res={}
gen = DI.DeckGenerator(index)

# C1 — role-tag coverage among trainers
trainers=set(index.trainers); roles=['draw_search','gust_switch','energy_accel']
tagged=set()
for r in roles: tagged |= set(index.role.get(r,[]))
cov=len(trainers & tagged)/max(len(trainers),1)
print(f'C1 role-tag coverage: {len(trainers & tagged)}/{len(trainers)} trainers tagged = {cov:.0%}')
for r in roles:
    print(f'      {r:13s}: {len(set(index.role.get(r,[])) & trainers):4d} trainers')
res['tag_coverage']=cov
res['tag_role_counts']={r:len(set(index.role.get(r,[])) & trainers) for r in roles}

# C2 — recipe sum + energy skew
recipe_sum=sum(t for _,t in DI._RECIPE); pad=DI.DECK_SIZE-recipe_sum
real_energy=[sum(n for c,n in Counter(d).items() if index.is_basic_energy(c)) for _,d in POOL]
map_guess=gen.complete(Counter(), rng=None)
map_energy=sum(n for c,n in map_guess.items() if index.is_basic_energy(c))
print(f'C2 recipe targets sum to {recipe_sum} -> padding fills {pad}/60 (energy-first)')
print(f'      empty-knowledge guess energy={map_energy} | real decks energy mean={np.mean(real_energy):.1f} '
      f'[{min(real_energy)}-{max(real_energy)}]')
res['recipe_sum']=recipe_sum; res['pad']=pad
res['map_energy']=int(map_energy); res['real_energy_mean']=float(np.mean(real_energy))

# C3 — world diversity vs matcher confidence as more is revealed
lib=DI.library_from_pool(decks_dir='decks', pattern='*.csv')
def reveal(deck,k,rng):
    cards=list(Counter(deck).elements()); rng.shuffle(cards); return Counter(cards[:k])
rng=random.Random(SEED); rows=[]; sample=[d for _,d in POOL[:8]]
for k in (3,6,12,20):
    nds=[]; confs=[]
    for d in sample:
        known=reveal(d,k,rng)
        worlds={tuple(sorted(gen.complete(known,rng=random.Random(s)).items())) for s in range(8)}
        nds.append(len(worlds))
        _,conf,_=lib.classify(known); confs.append(conf)
    print(f'C3 k={k:2d}: distinct worlds/8 = {np.mean(nds):.1f} | matcher conf = {np.mean(confs):.2f}')
    rows.append({'k':k,'distinct_worlds':float(np.mean(nds)),'matcher_conf':float(np.mean(confs))})
res['diversity_vs_conf']=rows
RESULTS['C_stress']=res
json.dump(RESULTS, open('experiment_results.json','w'), indent=1)
print('\nC done.')

## 6. Experiment A — offline reconstruction (no games)

For each real deck and each *k*, reveal *k* random cards and ask each method to guess the rest.
Score the **hidden** portion only (the revealed cards are handed to every method, so no free credit):
* **hidden-recall** — fraction of the 60−*k* hidden cards the guess places (higher = better).
* **TV-dist** — total-variation between guessed and true count-vectors (lower = better).

Methods: **generator** (current), **pool_sample** (a real consistent deck — the cheap alternative),
**freq_fill** (seed + fill ∝ corpus frequency — the cheap *upgrade*), **uniform_fill** (floor),
**matcher** (archetype template only). If **generator** can't beat **pool_sample / freq_fill**, that
alone argues for going data-driven.

In [ ]:
banner('EXPERIMENT A: offline deck reconstruction (no games)')
gen = DI.DeckGenerator(index)
lib = DI.library_from_pool(decks_dir='decks', pattern='*.csv')

# corpus card frequency (for the freq-weighted baseline = the cheap upgrade)
FREQ=Counter()
for _,d in POOL:
    for c,n in Counter(d).items(): FREQ[c]+=n
freq_cards=list(FREQ.keys()); freq_w=[float(FREQ[c]) for c in freq_cards]

def cap(c): return DI.DECK_SIZE if index.is_basic_energy(c) else DI.MAX_COPIES
def legal_fill(known, weighted, rng):
    deck=Counter()
    for c,n in known.items(): deck[c]=min(n,cap(c))
    guard=0
    while sum(deck.values())<DI.DECK_SIZE and guard<100000:
        guard+=1
        c = freq_cards[rng.choices(range(len(freq_cards)),weights=freq_w)[0]] if weighted else rng.choice(freq_cards)
        if deck[c]<cap(c): deck[c]+=1
    if not any(index.is_basic_pokemon(c) for c in deck) and index.basic_pokemon:
        deck[rng.choice(index.basic_pokemon)]+=1
        while sum(deck.values())>DI.DECK_SIZE:
            drop=next((c for c in deck if not index.is_basic_pokemon(c) and deck[c]>0),None)
            if drop is None: break
            deck[drop]-=1
            if deck[drop]==0: del deck[drop]
    return deck
def pool_consistent(known, rng):
    cand=[d for _,d in POOL if all(Counter(d)[c]>=n for c,n in known.items())]
    return Counter(rng.choice(cand)) if cand else gen.complete(known, rng=rng)
def matcher_method(known):
    cards,_=lib.predict_decklist(known); return Counter(cards) if cards else Counter()
def methods(known, rng):
    return {'generator':    gen.complete(known, rng=rng),
            'pool_sample':  pool_consistent(known, rng),
            'freq_fill':    legal_fill(known, True, rng),
            'uniform_fill': legal_fill(known, False, rng),
            'matcher':      matcher_method(known)}
def reveal(deck,k,rng):
    cards=list(Counter(deck).elements()); rng.shuffle(cards); return Counter(cards[:k])
def score(true, known, guess):
    th=true-known; gh=guess-known
    recall=sum((th & gh).values())/max(sum(th.values()),1)
    tv=0.5*sum(abs(true.get(c,0)-guess.get(c,0)) for c in set(true)|set(guess))/DI.DECK_SIZE
    return recall, tv

mnames=['generator','pool_sample','freq_fill','uniform_fill','matcher']
rng=random.Random(SEED); A={}
for k in RECON_REVEAL_KS:
    agg={m:{'recall':[],'tv':[]} for m in mnames}
    for _,deck in POOL:
        true=Counter(deck)
        for _ in range(RECON_REPS):
            known=reveal(deck,k,rng); ms=methods(known,rng)
            for m in mnames:
                r,tv=score(true,known,ms[m]); agg[m]['recall'].append(r); agg[m]['tv'].append(tv)
    print(f'\n  k={k:2d} revealed  (n={len(POOL)*RECON_REPS} trials/method)')
    print(f'    {"method":13s} {"hidden-recall":>13s} {"TV-dist":>9s}')
    A[str(k)]={}
    for m in mnames:
        rec=float(np.mean(agg[m]['recall'])); tv=float(np.mean(agg[m]['tv']))
        A[str(k)][m]={'recall':rec,'tv':tv}
        print(f'    {m:13s} {rec:12.1%}  {tv:8.3f}')
RESULTS['A_reconstruction']=A
json.dump(RESULTS, open('experiment_results.json','w'), indent=1)
print('\nA done. (key contrast: generator vs pool_sample / freq_fill)')

## 7. Experiment B — in-search ablation (decisive)

Swap **only** the determinization source inside ISMCTS and measure the net's win-rate vs the fixed
heuristic reference, on the **same** deck-pairs + seats across all conditions (paired ⇒ low variance):

| kind | opponent-deck guess | meaning |
|---|---|---|
| `none` | engine's all-basic fallback | floor |
| `current` | `DeckGenerator` (today) | what we ship |
| `pool` | a real consistent pool deck | cheap alternative |
| `oracle` | the opponent's **true** deck | upper bound |

**`oracle − current` is the headroom.** Small/within-CI ⇒ a better deck model (NN or otherwise)
can't move play strength; large ⇒ deck inference is worth investing in. **`current − none`** is what
the generator already buys. Needs a trained net.

In [ ]:
banner('EXPERIMENT B: determinization ablation (in-search win-rate)')
if NET is None:
    print('SKIP B — no net checkpoint attached. Attach hakase0/tcg-pokebot-weights and re-run.')
else:
    class _FixedKnown:
        def __init__(self, c): self._c=c
        def known(self): return self._c
    libB=DI.library_from_pool(decks_dir='decks', pattern='*.csv')

    def build_variant(kind, true_opp):
        # returns (update_fn, predict_fn|None) for one game
        if kind=='none':
            return (lambda o: None), None
        trk=DI.OpponentTracker()
        if kind=='current':
            return (lambda o: trk.update(o)), \
                   (lambda o,rng=None: DI.predict_opponent_zones(o,trk,libB,card_db=db,min_conf=0.3,rng=rng,index=index))
        if kind=='oracle':
            fixed=_FixedKnown(Counter(true_opp))
            return (lambda o: None), \
                   (lambda o,rng=None: DI.predict_opponent_zones(o,fixed,libB,card_db=db,min_conf=0.3,rng=rng,index=index))
        if kind=='pool':
            def pred(o,rng=None):
                known=trk.known()
                cand=[d for _,d in POOL if all(Counter(d)[c]>=n for c,n in known.items())]
                src=Counter((rng.choice(cand) if rng else random.choice(cand))) if cand else None
                tk=_FixedKnown(src) if src is not None else trk
                return DI.predict_opponent_zones(o,tk,libB,card_db=db,min_conf=0.3,rng=rng,index=index)
            return (lambda o: trk.update(o)), pred
        raise ValueError(kind)

    def make_agent(our_deck, kind, true_opp):
        def factory():
            upd,pred=build_variant(kind, true_opp)
            def policy(obs):
                sel=obs.get('select')
                if sel is None: return our_deck
                O=len(sel.get('option',[]))
                if (sel.get('minCount',1) or 0)>1 or O==0: return H.select(obs,db=db,attack_db=atk)
                upd(obs)
                if O==1: return [0]
                _,_,ch=ismcts.search(obs,our_deck,db,atk,NET,DEV,pred,
                                     n_worlds=ABLATION_WORLDS,n_sims=ABLATION_SIMS,leaf_eval=ABLATION_LEAF)
                if ch is not None: return [ch]
                p,_,_=arena._infer(NET,fx.encode_observation(obs,attack_lookup=atk),DEV)
                return [int(np.argmax(p[:O]))]
            return policy
        return factory

    # shared (our_deck, opp_deck, seat) across all kinds -> paired comparison
    rng=random.Random(SEED)
    pairs=[(rng.choice(POOL)[1], rng.choice(POOL)[1], i%2) for i in range(ABLATION_PAIRS)]
    B={}
    for kind in ABLATION_KINDS:
        w=n=0; t0=time.time()
        for gi,(da,dbk,seatA) in enumerate(pairs):
            res=arena.play_one(make_agent(da,kind,dbk)(), arena.make_heuristic_agent(db,atk,dbk)(),
                               da, dbk, seatA)
            if   res==seatA:        w+=1
            elif res==2 or res<0:   w+=0.5
            n+=1
            if (gi+1)%10==0: print(f'      {kind} {gi+1}/{len(pairs)} ... wr~{w/n:.0%}', flush=True)
        wr=w/max(n,1); _,lo,hi=arena.wilson(int(round(w)),n)
        B[kind]={'winrate':wr,'ci_low':lo,'ci_high':hi,'games':n}
        print(f'  {kind:8s}: net winrate {wr:.1%} [{lo:.0%},{hi:.0%}]  ({time.time()-t0:.0f}s)')
    RESULTS['B_ablation']={'worlds':ABLATION_WORLDS,'sims':ABLATION_SIMS,'leaf':ABLATION_LEAF,'by_kind':B}
    json.dump(RESULTS, open('experiment_results.json','w'), indent=1)
    if {'oracle','current','none'} <= set(B):
        head=B['oracle']['winrate']-B['current']['winrate']
        val =B['current']['winrate']-B['none']['winrate']
        print(f"\n  HEADROOM (oracle - current) = {head:+.1%}  <- max a perfect deck model could add")
        print(f"  VALUE    (current - none)   = {val:+.1%}  <- what the generator already buys")
        print('  -> headroom small/within-CI => a NN cannot help much; spend effort elsewhere.')

## 8. Summary

In [ ]:
banner('SUMMARY')
R=json.load(open('experiment_results.json'))
if 'C_stress' in R:
    c=R['C_stress']; print(f"C  tag-coverage {c['tag_coverage']:.0%} | recipe {c['recipe_sum']}/60 (+{c['pad']} pad) | "
                           f"empty-guess energy {c['map_energy']} vs real {c['real_energy_mean']:.1f}")
if 'A_reconstruction' in R:
    print('A  hidden-recall by method:')
    a=R['A_reconstruction']
    for k in sorted(a, key=int):
        order=sorted(a[k].items(), key=lambda kv:-kv[1]['recall'])
        print(f'    k={k:>2}: ' + '  '.join(f'{m}={v["recall"]:.0%}' for m,v in order))
if 'B_ablation' in R:
    b=R['B_ablation']['by_kind']
    print('B  net win-rate by determinization:')
    for kind in ('none','current','pool','oracle'):
        if kind in b: print(f"    {kind:8s} {b[kind]['winrate']:.0%} [{b[kind]['ci_low']:.0%},{b[kind]['ci_high']:.0%}]")
    if {'oracle','current'} <= set(b):
        print(f"    => headroom (oracle-current) {b['oracle']['winrate']-b['current']['winrate']:+.1%}")
else:
    print('B  skipped (no net).')
print('\nwrote experiment_results.json')